# Delhi Air Pollution — Data Visualization

This notebook explores `combined_delhi.csv` and generates the chart images in this folder.

**Important dataset limitations:** the supplied file contains only one city (`Delhi`) and one region (`North`), and it has no latitude/longitude columns. Therefore, a city ranking, state-vs-pollutant heatmap, and geographic map cannot be truthfully produced from this file. This notebook uses clearly labeled substitutes: highest-AQI days, month/hour heatmap, and wind-speed-vs-AQI scatter plot.

In [ ]:
import pandas as pd\nimport numpy as np\nimport matplotlib.pyplot as plt\nfrom pathlib import Path\n\nDATA_PATH = Path('combined_delhi.csv')  # Put the CSV beside this notebook\nOUT_DIR = Path('visualizations')\nOUT_DIR.mkdir(exist_ok=True)\ndf = pd.read_csv(DATA_PATH)\ndf['time'] = pd.to_datetime(df['time'], errors='coerce')\nprint('Shape:', df.shape)\nprint('Cities:', df['city'].dropna().unique())\nprint('Regions:', df['region'].dropna().unique())\nprint('Latitude/longitude columns present:', any(c in df.columns for c in ['latitude','longitude','lat','lon']))

## 1. Top 10 highest-AQI days

A city comparison is not possible because only Delhi is present. This chart instead ranks the ten dates with the highest mean AQI.

In [ ]:
daily = df.dropna(subset=['time','aqi']).assign(date=lambda x: x['time'].dt.date).groupby('date')['aqi'].mean().sort_values(ascending=False).head(10).sort_values()\nplt.figure(figsize=(10,6)); plt.barh([str(x) for x in daily.index], daily.values)\nplt.title('Top 10 Days by Average AQI — Delhi'); plt.xlabel('Mean AQI'); plt.ylabel('Date'); plt.tight_layout(); plt.show()

## 2. Month/hour AQI heatmap

In [ ]:
t = df.dropna(subset=['time','aqi']).copy(); t['month'] = t['time'].dt.month; t['hour'] = t['time'].dt.hour\np = t.pivot_table(index='month', columns='hour', values='aqi', aggfunc='mean').reindex(index=range(1,13), columns=range(24))\nplt.figure(figsize=(12,5)); im=plt.imshow(p.values, aspect='auto', origin='lower'); plt.colorbar(im,label='Mean AQI')\nplt.xticks(range(24),range(24)); plt.yticks(range(12),['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])\nplt.title('Average AQI by Month and Hour — Delhi'); plt.xlabel('Hour'); plt.ylabel('Month'); plt.tight_layout(); plt.show()

## 3. Pollutant boxplots

In [ ]:
pollutants = [c for c in ['pm2_5','pm10','co','no2','so2','ozone'] if c in df.columns and df[c].notna().any()]\nfig, axes = plt.subplots(2,3,figsize=(12,7))\nfor ax,col in zip(axes.flat,pollutants): ax.boxplot(df[col].dropna()); ax.set_title(col.upper()+' distribution'); ax.set_ylabel('Recorded value'); ax.set_xticks([])\nfor ax in axes.flat[len(pollutants):]: ax.axis('off')\nplt.suptitle('Pollutant Distributions and Outliers — Delhi'); plt.tight_layout(); plt.show()

## 4. AQI category share

In [ ]:
bins=[-np.inf,50,100,200,300,400,np.inf]; labels=['Good (0–50)','Satisfactory (51–100)','Moderate (101–200)','Poor (201–300)','Very Poor (301–400)','Severe (>400)']\ncats=pd.cut(df['aqi'].dropna(),bins=bins,labels=labels,right=True,include_lowest=True); counts=cats.value_counts().reindex(labels,fill_value=0); counts=counts[counts>0]\nplt.figure(figsize=(8,7)); plt.pie(counts.values,labels=counts.index,autopct='%1.1f%%',startangle=90); plt.title('Recorded AQI Observations by Category — Delhi'); plt.show()

## 5. Scatter plot: wind speed vs AQI

In [ ]:
s=df.dropna(subset=['wind_speed','aqi']); plt.figure(figsize=(9,6)); plt.scatter(s['wind_speed'],s['aqi'],alpha=.25,s=10)\nplt.title('Wind Speed vs AQI — Delhi'); plt.xlabel('Wind speed (dataset units)'); plt.ylabel('AQI'); plt.tight_layout(); plt.show()

## 6. Monthly pollutant trend index

In [ ]:
monthly=df.dropna(subset=['time']).set_index('time')[pollutants].resample('MS').mean(); indexed=monthly/monthly.mean()*100\nplt.figure(figsize=(11,6))\nfor col in pollutants: plt.plot(indexed.index,indexed[col],marker='o',label=col.upper())\nplt.title("Monthly Pollutant Trend Index (each pollutant's mean = 100)"); plt.xlabel('Month'); plt.ylabel('Index'); plt.legend(ncol=3); plt.tight_layout(); plt.show()

## Interpretation and limitations

- The file has 36,456 rows and 16 columns; it records Delhi/North only.
- AQI/pollutant observations can show temporal patterns, distribution and outliers.
- A real map or latitude-longitude scatter plot requires coordinate columns or a separate station-location dataset.
- A top-10 city chart and state-vs-pollutant heatmap require data from multiple cities/states; do not interpret the substitutes as city/state comparisons.